In [1]:
import re
import numpy as np
import pandas as pd

GROUPES = ["cs", "math", "physics", "q-bio", "q-fin", "stat", "eess", "econ"]

df = pd.read_csv("../data/raw/arxiv_echantillon.csv", dtype={"id": str})
print("Taille :", df.shape)

Taille : (17349, 11)


In [2]:
def nettoyer(texte):
    texte = texte.replace(r"\$", " dollars ")                            # \$100 -> dollars 100
    texte = re.sub(r"\$\$.*?\$\$", " formula ", texte, flags=re.DOTALL)  # $$ ... $$ -> formula
    texte = re.sub(r"\$.*?\$", " formula ", texte, flags=re.DOTALL)      # $ ... $   -> formula
    texte = re.sub(r"\\[a-zA-Z]+", " ", texte)                           # \emph, \cite... -> supprimés
    texte = re.sub(r"\\[^a-zA-Z\s]", "", texte)                          # accents : \" \' -> supprimés
    texte = texte.replace("\\", " ")                                     # backslashs restants
    texte = re.sub(r"[{}]", "", texte)                                   # accolades { }
    texte = re.sub(r"\s+", " ", texte)                                   # \n et espaces multiples -> 1 espace
    return texte.strip()                                                 # espaces au début et à la fin


# Test sur une phrase inventée
test = r'We study $x^2$ and \emph{graphs}, costing \$100, in Schr\"odinger systems.'
print("AVANT :", test)
print("APRÈS :", nettoyer(test))

AVANT : We study $x^2$ and \emph{graphs}, costing \$100, in Schr\"odinger systems.
APRÈS : We study formula and graphs, costing dollars 100, in Schrodinger systems.


In [3]:
# On applique nettoyer() à chaque résumé -> nouvelle colonne "text"
df["text"] = df["abstract"].apply(nettoyer)

# Un exemple avant / après
print("AVANT :", repr(df["abstract"].iloc[1][:250]))
print()
print("APRÈS :", repr(df["text"].iloc[1][:250]))
print()

# Vérifications
print("Résumés qui contenaient des formules :", df["abstract"].str.contains("$", regex=False).sum())
print("Backslashs restants :", df["text"].str.contains("\\", regex=False).sum())
print("Textes vides        :", (df["text"].str.len() == 0).sum())
print("Textes en double    :", df["text"].duplicated().sum())

AVANT : "  Motivated by the very recent measurement of the branching ratio of ${B_d^0}\n\\to J/\\psi \\eta$ decay, we calculate the branching ratios of ${B_d}^0 \\to\nJ/\\psi \\eta$ and ${B_d}^0 \\to J/\\Psi \\eta'$ decays in the perturbative QCD\n(pQCD) approach. The pQ"

APRÈS : 'Motivated by the very recent measurement of the branching ratio of formula decay, we calculate the branching ratios of formula and formula decays in the perturbative QCD (pQCD) approach. The pQCD predictions for the branching ratios of considered dec'

Résumés qui contenaient des formules : 3258
Backslashs restants : 0
Textes vides        : 0
Textes en double    : 0


In [4]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

Y = df[GROUPES].values  # tableau de 17 349 lignes x 8 colonnes (les 0 et 1)

# 1er découpage : 70 % train / 30 % "reste"
decoupe1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
idx_train, idx_reste = next(decoupe1.split(np.zeros(len(df)), Y))
df_train = df.iloc[idx_train]
df_reste = df.iloc[idx_reste]

# 2e découpage : le "reste" coupé en deux -> 15 % validation / 15 % test
decoupe2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
idx_val, idx_test = next(decoupe2.split(np.zeros(len(df_reste)), df_reste[GROUPES].values))
df_val = df_reste.iloc[idx_val]
df_test = df_reste.iloc[idx_test]

print("Train :", len(df_train), "| Validation :", len(df_val), "| Test :", len(df_test))

Train : 12132 | Validation : 2609 | Test : 2608


In [5]:
proportions = pd.DataFrame({
    "train %": 100 * df_train[GROUPES].mean(),
    "val %":   100 * df_val[GROUPES].mean(),
    "test %":  100 * df_test[GROUPES].mean(),
}).round(1)
print(proportions)
print()
print("Articles communs entre train et test :", len(set(df_train["id"]) & set(df_test["id"])))

         train %  val %  test %
cs          39.3   39.2    39.2
math        23.3   23.2    23.2
physics     24.0   23.9    23.9
q-bio       15.0   14.9    14.9
q-fin       16.8   16.7    16.8
stat        18.6   18.6    18.6
eess        14.8   14.7    14.7
econ        14.1   14.0    14.0

Articles communs entre train et test : 0


In [6]:
COLONNES = ["id", "title", "text"] + GROUPES  # on garde le texte nettoyé + les 8 labels

df_train[COLONNES].to_csv("../data/processed/train.csv", index=False)
df_val[COLONNES].to_csv("../data/processed/val.csv", index=False)
df_test[COLONNES].to_csv("../data/processed/test.csv", index=False)

print("3 fichiers sauvegardés dans data/processed/")

3 fichiers sauvegardés dans data/processed/
